In [3]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [4]:
from __future__ import annotations

import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any, ClassVar

import pandas as pd
from openpyxl import load_workbook
from openpyxl.worksheet.worksheet import Worksheet


class HotelExcelFlattener:
    """
    Transforme la première feuille du fichier Excel ROD en DataFrame tabulaire :
    - 1 ligne par hôtel,
    - 1 colonne par champ source,
    - toutes les colonnes descriptives B:J sont injectées dans le nom du champ,
    - les cellules calculées par formule Excel sont conservées avec le suffixe '__formula'.

    Exemple de nom généré :
    etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h__bdd_accor_source__360_hotel_referential
    """

    HEADER_ROW: ClassVar[int] = 3

    # Colonnes descriptives du champ source : B à J
    FIELD_METADATA_START_COL: ClassVar[int] = 2
    FIELD_METADATA_END_COL: ClassVar[int] = 10

    # Colonnes hôtels : K à Q
    HOTEL_START_COL: ClassVar[int] = 11
    HOTEL_END_COL: ClassVar[int] = 17

    SEPARATOR: ClassVar[str] = "__"
    FORMULA_SUFFIX: ClassVar[str] = "__formula"
    SOURCE_ROW_SUFFIX: ClassVar[str] = "source_row"

    @classmethod
    def to_dataframe(
        cls,
        excel_path: str | Path,
        sheet_name: str | None = None,
        *,
        include_source_row_when_duplicate: bool = True,
        include_hotel_metadata: bool = True,
    ) -> pd.DataFrame:
        """
        Lit le fichier Excel et retourne une DataFrame aplatie.

        Parameters
        ----------
        excel_path:
            Chemin vers le fichier Excel.
        sheet_name:
            Nom de la feuille. Si None, utilise la première feuille.
        include_source_row_when_duplicate:
            Si deux lignes Excel produisent le même nom de champ, ajoute '__source_row__<num_ligne>'
            pour éviter l'écrasement.
        include_hotel_metadata:
            Ajoute des colonnes d'identification de l'hôtel, par exemple hotel__name et hotel__brand.

        Returns
        -------
        pd.DataFrame
            DataFrame avec 1 ligne par hôtel.
        """
        excel_path = Path(excel_path)

        wb_values = load_workbook(excel_path, data_only=True)
        wb_formulas = load_workbook(excel_path, data_only=False)

        ws_values = cls._get_sheet(wb_values, sheet_name)
        ws_formulas = cls._get_sheet(wb_formulas, sheet_name)

        metadata_headers = cls._read_metadata_headers(ws_values)
        hotel_columns = cls._detect_hotel_columns(ws_values)
        field_specs = cls._build_field_specs(
            ws_values=ws_values,
            ws_formulas=ws_formulas,
            metadata_headers=metadata_headers,
            include_source_row_when_duplicate=include_source_row_when_duplicate,
        )

        rows: list[dict[str, Any]] = []

        for hotel in hotel_columns:
            row: dict[str, Any] = {}

            if include_hotel_metadata:
                row[f"hotel{cls.SEPARATOR}name"] = hotel["name"]
                if cls._has_value(hotel.get("brand")):
                    row[f"hotel{cls.SEPARATOR}brand"] = hotel["brand"]
            else:
                row["hotel"] = hotel["name"]

            for spec in field_specs:
                value = ws_values.cell(row = spec["row_idx"], column = hotel["col_idx"]).value
                row[spec["field_name"]] = value

            rows.append(row)

        return pd.DataFrame(rows)

    @classmethod
    def _get_sheet(cls, workbook, sheet_name: str | None) -> Worksheet:
        return workbook[sheet_name] if sheet_name else workbook[workbook.sheetnames[0]]

    @classmethod
    def _read_metadata_headers(cls, ws: Worksheet) -> dict[int, str]:
        """Lit les intitulés des colonnes descriptives B:J sur HEADER_ROW."""
        headers: dict[int, str] = {}

        for col_idx in range(cls.FIELD_METADATA_START_COL, cls.FIELD_METADATA_END_COL + 1):
            raw_header = ws.cell(row = cls.HEADER_ROW, column = col_idx).value
            if cls._has_value(raw_header):
                headers[col_idx] = cls._clean_text(raw_header)
            else:
                headers[col_idx] = f"metadata_col_{col_idx}"

        return headers

    @classmethod
    def _detect_hotel_columns(cls, ws: Worksheet) -> list[dict[str, Any]]:
        """
        Détecte les colonnes hôtels K:Q.
        La ligne 3 contient le nom de l'hôtel.
        La ligne 2 contient parfois la marque, avec cellules fusionnées ; on fait donc un forward-fill.
        """
        hotels: list[dict[str, Any]] = []
        current_brand: str | None = None

        for col_idx in range(cls.HOTEL_START_COL, cls.HOTEL_END_COL + 1):
            raw_brand = ws.cell(row = cls.HEADER_ROW - 1, column = col_idx).value
            if cls._has_value(raw_brand):
                current_brand = str(raw_brand).strip()

            raw_hotel_name = ws.cell(row = cls.HEADER_ROW, column=col_idx).value
            if not cls._has_value(raw_hotel_name):
                continue

            hotels.append(
                {
                    "col_idx": col_idx,
                    "name": str(raw_hotel_name).strip(),
                    "brand": current_brand,
                }
            )

        if not hotels:
            raise ValueError(
                "Aucune colonne hôtel détectée. Vérifie HOTEL_START_COL, HOTEL_END_COL et HEADER_ROW."
            )

        return hotels

    @classmethod
    def _build_field_specs(
        cls,
        *,
        ws_values: Worksheet,
        ws_formulas: Worksheet,
        metadata_headers: dict[int, str],
        include_source_row_when_duplicate: bool,
    ) -> list[dict[str, Any]]:
        """
        Construit les noms de champs à partir de toutes les métadonnées B:J.
        Les valeurs vides des colonnes hiérarchiques sont forward-fill pour conserver le contexte.
        """
        specs: list[dict[str, Any]] = []
        forward_filled_metadata: dict[int, Any] = {}

        for row_idx in range(cls.HEADER_ROW + 1, ws_values.max_row + 1):
            parts: list[str] = []

            for col_idx in range(cls.FIELD_METADATA_START_COL, cls.FIELD_METADATA_END_COL + 1):
                raw_value = ws_values.cell(row = row_idx, column = col_idx).value

                if cls._has_value(raw_value):
                    forward_filled_metadata[col_idx] = raw_value

                # Pour les colonnes descriptives B:J, les cellules vides veulent souvent dire
                # "même valeur que la ligne précédente". On utilise donc la dernière valeur connue.
                value = forward_filled_metadata.get(col_idx)

                if not cls._has_value(value):
                    continue

                header = metadata_headers[col_idx]
                parts.extend([header, cls._clean_text(value)])

            if not parts:
                continue

            base_field_name = cls.SEPARATOR.join(parts)

            # Si au moins une cellule hôtel de cette ligne est une formule, le champ est marqué comme formule.
            if cls._row_contains_formula(ws_formulas, row_idx):
                base_field_name = f"{base_field_name}{cls.FORMULA_SUFFIX}"

            specs.append(
                {
                    "row_idx": row_idx,
                    "field_name": base_field_name,
                }
            )

        if include_source_row_when_duplicate:
            specs = cls._deduplicate_field_names(specs)

        return specs

    @classmethod
    def _row_contains_formula(cls, ws_formulas: Worksheet, row_idx: int) -> bool:
        """Retourne True si au moins une cellule hôtel de la ligne contient une formule Excel."""
        for col_idx in range(cls.HOTEL_START_COL, cls.HOTEL_END_COL + 1):
            value = ws_formulas.cell(row=row_idx, column=col_idx).value
            if cls._is_formula(value):
                return True
        return False

    @classmethod
    def _deduplicate_field_names(cls, specs: list[dict[str, Any]]) -> list[dict[str, Any]]:
        """Évite qu'une colonne écrase une autre si deux lignes génèrent le même nom."""
        counts = Counter(spec["field_name"] for spec in specs)

        for spec in specs:
            if counts[spec["field_name"]] > 1:
                spec["field_name"] = (
                    f"{spec['field_name']}"
                    f"{cls.SEPARATOR}{cls.SOURCE_ROW_SUFFIX}"
                    f"{cls.SEPARATOR}{spec['row_idx']}"
                )

        return specs

    @classmethod
    def _is_formula(cls, value: Any) -> bool:
        return isinstance(value, str) and value.startswith("=")

    @classmethod
    def _has_value(cls, value: Any) -> bool:
        return value is not None and str(value).strip() != ""

    @classmethod
    def _clean_text(cls, value: Any) -> str:
        """
        Nettoie une valeur pour l'utiliser dans un nom de colonne stable :
        - minuscules,
        - accents retirés,
        - espaces et ponctuation remplacés par '_',
        - underscores multiples compressés.
        """
        text = str(value).strip().lower().replace("\n", " ")
        text = unicodedata.normalize("NFKD", text)
        text = "".join(char for char in text if not unicodedata.combining(char))
        text = re.sub(r"\s+", "_", text)
        text = re.sub(r"[^a-zA-Z0-9_]+", "_", text)
        text = re.sub(r"_+", "_", text)
        return text.strip("_")


In [5]:
df = HotelExcelFlattener.to_dataframe(
    "Récapitulatif de l'ensemble des données ROD.xlsx"
)

In [6]:
X_cols = [col for col in df.columns if "X" in df[col].unique()]
for col in X_cols:
    df[col] = df[col].str.strip().replace("X", "OUI").replace("-", "NON")

In [7]:
simul_step_cols = [col for col in df.columns if col.startswith("etape_rod__5_simulateur_de_revenus")]
simul_step_cols

['etape_rod__5_simulateur_de_revenus__sous_etape_rod__ecran_de_controle_parametres__data__nb_de_chambres__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__oui__commentaire_thomas_innolab__voir_regle_n_1_dans_mes_3_simulateurs_excel',
 'etape_rod__5_simulateur_de_revenus__sous_etape_rod__ecran_de_controle_parametres__data__moyen_de_guests_par_chambre__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__tbc__bdd_accor_source__donnees_fictives_pour_le_moment__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__data_a_ajouter_au_futur_calcul_de_simulation',
 'etape_rod__5_simulateur_de_revenus__sous_etape_rod__ecran_de_controle_parametres__data__to_annuel_moyen__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__tbc__

In [8]:
formula_cols = [col for col in df.columns if col.endswith("formula")]
formula_cols

['etape_rod__5_simulateur_de_revenus__sous_etape_rod__mix_produits_choisi_par_l_hotel_dans_un_1er_temps_puis_option_bouton_reco_ia__data__de_produits_f_b__deja_dans_rod_au_moment_ou_le_user_se_connecte__non_mais_option_ai__accor_api_portfolio__non__bdd_accor_source__n_a__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__oui__commentaire_thomas_innolab__voir_regle_n_2_dans_mes_3_simulateurs_excel_application_d_un_coefficient_moyen_en_fonction_du_type_de_boutique_recommande_a_l_hotel__formula',
 'etape_rod__5_simulateur_de_revenus__sous_etape_rod__mix_produits_choisi_par_l_hotel_dans_un_1er_temps_puis_option_bouton_reco_ia__data__de_produits_non_f_b__deja_dans_rod_au_moment_ou_le_user_se_connecte__non_mais_option_ai__accor_api_portfolio__non__bdd_accor_source__n_a__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__oui__commentaire_thomas_innolab__voir_regle_n_2_dans_mes_3_simulateurs_excel_application_d_un_coefficient_

In [9]:
fictive_cols = [col for col in df.columns if "fictive" in col]
fictive_cols

['etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__derniere_reno_hotel__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__tbc__bdd_accor_source__donnees_fictives_pour_le_moment__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__',
 'etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__derniere_reno_lobby__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__tbc__bdd_accor_source__donnees_fictives_pour_le_moment__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__',
 'etape_rod__1_informations_generales__sous_etape_rod__donnees_chiffrees__data__adultes_par_chambre__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__tbc__bdd_accor_source__donnees_fictives_pour_le_moment__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_ac

In [10]:
trivial_columns = [col for col in df.columns if(len(set(df[col].unique()) - {"?", np.NAN, None})<= 1)]
len(trivial_columns)

57

In [11]:
adrs_cols = [col for col in df.columns if "adresse_postale" in col]
len(adrs_cols)

3

In [12]:
df

,hotel__name,hotel__brand,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h__deja_dans_rod_au_moment_ou_le_user_se_connecte__non__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__complete_par_le_user_au_moment_de_la_connexion,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__nom_de_l_hotel__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__non__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__remonte_automatiquement,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_1__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_2__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__adresse_postale_3__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__code_postal__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__ville__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__longitude__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__export_a_la_demande__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__latitude__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__export_a_la_demande__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__marque__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__non__data_exploitee_dans_la_sim

In [13]:
remove_cols = sorted(set(simul_step_cols + formula_cols + fictive_cols + trivial_columns + adrs_cols))
len(remove_cols)

88

In [14]:
len(df.columns)

136

In [16]:
keep_cols = sorted(set(df.columns) - set(remove_cols))
df_out = df[keep_cols].transpose()

In [19]:
df[keep_cols]

,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__code_h__deja_dans_rod_au_moment_ou_le_user_se_connecte__non__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__complete_par_le_user_au_moment_de_la_connexion,etape_rod__0_page_de_connexion__sous_etape_rod__id__data__nom_de_l_hotel__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__non__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__remonte_automatiquement,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__code_postal__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__latitude__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__export_a_la_demande__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__longitude__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__export_a_la_demande__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__0_page_de_connexion__sous_etape_rod__localisation_geo__data__ville__deja_dans_rod_au_moment_ou_le_user_se_connecte__n_a__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__n_a__data_exploitee_dans_la_simulation_actuellement__non__commentaire_thomas_innolab__pourrait_etre_utilise_par_l_ia_pour_affiner_la_recommandation_d_un_concept_de_corner_de_vente,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__contrat_signe_annee__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__tbc__bdd_accor_source____le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__contrat_type__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__tbc__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__dom_dof__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__tbc__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__oui__data_exploitee_dans_la_simulation_actuellement__n_a__commentaire_thomas_innolab__,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__marque__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__non__data_exploitee_dans_la_simulation_actuellement__oui__commentaire_thomas_innolab__voir_fichier_parametres_regles_projections_nb_d_hotels,etape_rod__1_informations_generales__sous_etape_rod__donnees_admin__data__nb_de_chambres__deja_dans_rod_au_moment_ou_le_user_se_connecte__oui__accor_api_portfolio__api_portfolio__bdd_accor_source__360_hotel_referential__le_user_peut_modifier_dans_rod__

In [ ]:
wanted_cols = [
    "hotel__name",
    "hotel__brand"
]